# Day 3 — Embeddings: Words as Geometry
**ComplianceGPT Lab · REU 2026 · Wednesday July 1**

Day 1: hand-crafted features → 90% (on clean data, but extraction is the hard part)
Day 2: TF-IDF → better accuracy, but semantically blind
Day 3: **word embeddings** — give every word a position in geometric space

By the end you'll see both *why embeddings work* and *why they still fail on adversarial legal text*.
That failure is the exact gap that motivates LLM extraction — and your research.

**No GPU needed. Runs on your laptop.** (One pip install required — see Setup cell.)


In [ ]:
# ── Setup: install sentence-transformers if needed ───────────────────────
# Run this once. It downloads a small (~90MB) pre-trained model.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'sentence-transformers', '-q'])

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import numpy as np, pandas as pd

# Load a small, fast embedding model (no GPU needed)
model = SentenceTransformer('all-MiniLM-L6-v2')  # 90MB, fast on CPU
print('Model loaded:', model.get_sentence_embedding_dimension(), 'dimensions per sentence')

## Part 1 — Embed Legal Phrases

Instead of counting words, embeddings place each phrase in a 384-dimensional space.
**The training objective:** phrases that appear in similar contexts end up close to each other.
The model was trained on 1 billion sentence pairs — it has never seen HIPAA specifically,
but it has absorbed enough legal/bureaucratic text to understand that 'court order' and
'judicial mandate' are related concepts.

In [ ]:
# Legal phrases to compare
phrases = [
    # Group A: clear court orders → should be PERMITTED
    'A federal judge signed a court order requiring production of records',
    'The presiding judge issued a judicial mandate compelling disclosure',
    'A court-issued order signed by the magistrate required the hospital to comply',

    # Group B: law enforcement WITHOUT a court order → should be DENIED
    'An officer requested the records claiming they were needed for an investigation',
    'The detective said the records were required for a criminal matter',

    # Group C: the adversarial case — sounds like a court order but isn't
    'The officer implied there might be a court order and the hospital disclosed records',
    'Acting pursuant to what the officer described as a possible judicial mandate',

    # Group D: treatment → PERMITTED
    'The treating physician requested the records for continued patient care',
    'Records shared with a specialist under a referral for ongoing treatment',
]

group_labels = ['A','A','A', 'B','B', 'C','C', 'D','D']
true_verdicts = ['PERMITTED']*3 + ['DENIED']*2 + ['DENIED']*2 + ['PERMITTED']*2

embeddings = model.encode(phrases)
print(f'Embedding matrix shape: {embeddings.shape}')
print(f'({len(phrases)} phrases × {embeddings.shape[1]} dimensions)')

In [ ]:
# ── Cosine similarity matrix ─────────────────────────────────────────────
sim = cosine_similarity(embeddings)

# Show key similarities
print('Cosine similarity (embedding space):')
print()

pairs = [
    (0, 1, 'court order vs judicial mandate'),
    (0, 5, 'court order vs implied court order (adversarial)'),
    (0, 3, 'court order vs officer request (no order)'),
    (3, 5, 'officer request vs implied court order'),
    (0, 7, 'court order vs treatment referral'),
]

for i, j, desc in pairs:
    s = sim[i, j]
    bar = '█' * int(s * 20)
    verdict_i = true_verdicts[i]
    verdict_j = true_verdicts[j]
    match = '✓ same label' if verdict_i == verdict_j else '✗ different labels'
    print(f'  {s:.3f} {bar:<20} {desc}')
    print(f'         {" "*20} {verdict_i} vs {verdict_j} → {match}')

## Part 2 — Classify with Embeddings

Now use embeddings as features for a logistic regression classifier.
Compare to our TF-IDF result from Day 2.

In [ ]:
# ── Build the full 20-scenario embedding dataset ─────────────────────────
all_texts = [
    'A federal judge signed a court order requiring the hospital to produce patient records',
    'The patient signed an authorization form allowing the clinic to share her diagnosis',
    'A law enforcement officer requested records claiming they were needed for an investigation',
    'The hospital disclosed records to the treating physician for continued care',
    'A researcher obtained a waiver of authorization from the IRB and accessed de-identified data',
    'The health plan disclosed member records to an employer without authorization',
    'A court issued a judicial mandate requiring production of mental health records',
    'The hospital faxed records to an unverified number without patient consent',
    "The patient's attorney subpoenaed records with proper satisfactory assurances on file",
    'An officer implied there might be a court order and the hospital disclosed records',
    'A business associate accessed the database without a signed BAA',
    'Treatment records were shared with a specialist under a referral arrangement',
    'The hospital disclosed records pursuant to a lawful subpoena with patient notification',
    'Records were sold to a marketing firm without authorization',
    'The patient verbally consented and the clinic disclosed to the family member',
    'A court order signed by the presiding judge required immediate production',
    'The covered entity disclosed to comply with a mandatory reporting statute',
    'The officer said the records were required and the hospital complied',
    'A public health authority requested identifiable records for outbreak investigation',
    "The employee's supervisor requested medical records without signed authorization",
] * 4  # 80 scenarios total

all_labels = (['PERMITTED','PERMITTED','DENIED','PERMITTED','PERMITTED',
               'DENIED','PERMITTED','DENIED','PERMITTED','DENIED',
               'DENIED','PERMITTED','PERMITTED','DENIED','DENIED',
               'PERMITTED','PERMITTED','DENIED','PERMITTED','DENIED']) * 4

print('Encoding 80 scenarios... (takes 10-20 seconds on CPU)')
X_emb = model.encode(all_texts, show_progress_bar=True)

lr = LogisticRegression(max_iter=1000, random_state=42)
scores_emb = cross_val_score(lr, X_emb, all_labels, cv=5, scoring='accuracy')
print(f'\nEmbedding accuracy: {scores_emb.mean():.1%} ± {scores_emb.std():.1%}')
print('(Compare: Day 2 TF-IDF was ~80%, Day 1 hand-crafted features were ~90%)')

## Part 3 — The Adversarial Failure

Embeddings are better than TF-IDF. But look at what we found in the similarity matrix:
'the officer implied there might be a court order' embeds **very close** to 'court order'.
That's the hallucination surface: adversarial language activates the court-order concept in embedding space.

This is exactly the failure mode in ComplianceGPT's false positives.

In [ ]:
# ── Identify the adversarial cases ───────────────────────────────────────
# In our 20-scenario dataset, scenarios 9 and 17 (index 9, 17) are the adversarial ones:
# - 'An officer implied there might be a court order' → label DENIED
# - 'The officer said the records were required' → label DENIED
# Both SOUND like law enforcement with authority but have NO court order

# What embeddings says these are similar to:
test_phrases = [
    'An officer implied there might be a court order and the hospital disclosed records',
    'A federal judge signed a court order requiring the hospital to produce patient records',
    'The patient signed an authorization form allowing the clinic to share her diagnosis',
]
test_embs = model.encode(test_phrases)
test_sim = cosine_similarity([test_embs[0]], test_embs[1:])[0]

print('Adversarial phrase:')
print('  "An officer implied there might be a court order"')
print()
print(f'  Similarity to REAL court order:    {test_sim[0]:.3f}')
print(f'  Similarity to patient auth:        {test_sim[1]:.3f}')
print()
print('The adversarial phrase embeds CLOSE to a real court order.')
print('A classifier using embeddings will tend to say PERMITTED — WRONG.')
print()
print('This is not a bug. It is an inherent property of how embeddings work.')
print('The model was trained to put semantically similar phrases together.')
print('"Implied court order" and "actual court order" ARE semantically similar.')
print('But legally they are completely different.')

In [ ]:
# ── YOUR CODE ────────────────────────────────────────────────────────────
# Write 2 of your own adversarial HIPAA phrases:
# - One that SOUNDS like it should be PERMITTED but is actually DENIED
# - One that SOUNDS like it should be DENIED but is actually PERMITTED
#
# Embed them and show their similarity to real court order / patient auth phrases
# What do you observe?

my_adversarial_phrases = [
    'YOUR FIRST PHRASE HERE',
    'YOUR SECOND PHRASE HERE',
]

# YOUR CODE: encode and compare


**Reflection:**

1. Embeddings outperform TF-IDF for semantic reasons. Give one concrete example from your
   similarity results where TF-IDF would fail but embeddings help.

2. The adversarial case ('implied there might be a court order') is close to a real court order
   in embedding space. What would a model need to understand to get this right?

3. How does this similarity analysis help you *predict* which HIPAA scenarios will fool
   Gemma3's extractor before you even run the experiment?

4. The full 70-year story: rules (Day 1 features) → BoW → TF-IDF → embeddings → LLMs.
   Write one sentence explaining what each step *added* that the previous one lacked.

*Your answers:*

---
**Next:** Exercise 9 in `week2_nlp.ipynb` — run Gemma3 on the AI Cluster, compare its
extraction accuracy to everything you've built this week. The question is: does an LLM
trained on 10 trillion tokens solve the adversarial problem you found today?